# Evaluación de Flujos Text2Midi y MidiLLM

Este notebook es parte de la investigación académica de maestría. Su propósito es evaluar el flujo completo de generación de MIDI desde descripciones naturales, comparando las dos estrategias principales:
1. **Text2Midi** (Progressive Search / Step-by-Step) usando el perfil `BALANCED`.
2. **MidiLLM** (Batch Generation / Best-of-N Search) usando el perfil `MIDILLM_FAST`.

El objetivo es documentar la traducción (technical prompt) resultante y el MIDI generado final para un objetivo específico.

## 1. Configuración de dependencias y entorno

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Obtener el directorio raíz del proyecto detectando la carpeta 'src'
current_dir = Path.cwd()
if (current_dir / "src").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists():
    project_root = current_dir.parent
else:
    # Si se ejecuta como script .py
    project_root = (
        Path(__file__).resolve().parent.parent
        if "__file__" in globals()
        else current_dir
    )

# Cambiar el directorio de trabajo a la raíz para que las rutas relativas (como models/) funcionen correctamente
if Path.cwd() != project_root:
    os.chdir(project_root)
    print(f"Directorio de trabajo cambiado a: {project_root}")

# Asegurar que importamos desde src
sys.path.append(str(project_root / "src"))

load_dotenv()  # Carga de GOOGLE_API_KEY desde .env

# Verificar la clave
if not os.getenv("GOOGLE_API_KEY"):
    print(
        "⚠️ ADVERTENCIA: No se encontró GOOGLE_API_KEY en el entorno. La traducción de LLM no funcionará correctamente."
    )

Directorio de trabajo cambiado a: /home/nickescolr/Maestria/proyecto


## 2. Importar componentes del Pipeline

In [2]:
# Configurar logs para ver el proceso
import logging

from adapters.translators.google_ai_translator import GoogleAIConfig
from config.profiles import get_profile
from pipeline import Text2MidiPipeline

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

## 3. Inicializar Pipeline

Instanciamos el pipeline inyectando la configuración para el traductor. Los modelos generadores (Text2Midi y MidiLLM) se inicializan bajo demanda según el perfil elegido.

In [3]:
# Usar Gemini 2.5 Flash u otro modelo para la traducción
translator_config = GoogleAIConfig(model_name="gemma-4-26b-a4b-it", temperature=0.9)

print("Inicializando pipeline...")
pipeline = Text2MidiPipeline(translator_config=translator_config)
print("✅ Pipeline inicializado.")

2026-05-25 16:47:38,579 - pipeline - INFO - Initializing Text2MidiPipeline...
2026-05-25 16:47:38,589 - pipeline - INFO - Pipeline initialization complete


Inicializando pipeline...
✅ Pipeline inicializado.


## 4. Definición de la Prueba Base

Definimos el prompt natural que servirá como base para ambas pruebas.

In [4]:
# Prompt natural del usuario / investigador
user_prompt = "Una melodía de piano melancólica y lenta, con un acompañamiento suave de cuerdas de fondo."

# Directorio de salida
output_dir = Path("outputs/evaluacion-notebook")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"📝 Prompt de prueba: '{user_prompt}'")

📝 Prompt de prueba: 'Una melodía de piano melancólica y lenta, con un acompañamiento suave de cuerdas de fondo.'


## 5. Prueba 1: Flujo Text2Midi (Progressive Search)

Utilizaremos el perfil `balanced`, que usa beam search y evaluación progresiva con heurísticas y CLAP.

In [5]:
print("\n" + "=" * 50)
print("🎹 INICIANDO PRUEBA 1: Text2Midi (Progressive Search)")
print("=" * 50)

# Cargar el perfil más rápido
profile_t2m = get_profile("one-shot")

# Generar MIDI
result_t2m = pipeline.generate(text=user_prompt, profile=profile_t2m)

print("\n✨ Resultados Text2Midi:")
print(
    f"- Traducción (Technical Prompt) usada por el generador:\n{result_t2m.technical_prompt}"
)
print(f"- Tamaño del MIDI generado: {len(result_t2m.midi_bytes)} bytes")

# Guardar
out_path_t2m = output_dir / "text2midi_resultado.mid"
with open(out_path_t2m, "wb") as f:
    f.write(result_t2m.midi_bytes)
print(f"💾 Guardado en: {out_path_t2m}")

2026-05-25 16:29:38,650 - pipeline - INFO - Generating MIDI for: 'Una melodía de piano melancólica y lenta, con un a...' with profile GenerationProfile(token_batch_size=300, num_beams=2, top_k=1, max_tokens=600, random_seed=None, clap_weight=0.3, key_weight=0.35, note_weight=0.35, strict_instruments=False, generator_type='text2midi', num_outputs=1)
2026-05-25 16:29:38,754 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 16:29:38,768 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/tokenizer_config.json "HTTP/1.1 200 OK"



🎹 INICIANDO PRUEBA 1: Text2Midi (Progressive Search)


2026-05-25 16:29:38,882 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-25 16:29:38,980 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-25 16:29:39,738 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 16:29:39,752 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"
2026-05-25 16:29:39,884 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/111 [00:00<?, ?it/s]

[transformers] T5EncoderModel LOAD REPORT from: google/flan-t5-base
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-25 16:29:40,281 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-05-25 16:30:11,971 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-26b-a4b-it:generateContent "HTTP/1.1 200 OK"
2026-05-25 16:30:11,974 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-05-25 16:30:43,961 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-26b-a4b-it:generateContent "HTTP/1.1 200 OK"
2026-05-25 16:30:43,963 - use_cases.progressive_search - INFO - Starting beam search with 2 branches, max_tokens=600
2026-05-25 16:31:43,565 - use_cases.progressive_search - INF


✨ Resultados Text2Midi:
- Traducción (Technical Prompt) usada por el generador:
A cinematic neoclassical track with a deeply melancholic and reflective vibe, featuring a soft felt piano performing a delicate, slow melody supported by a gentle, atmospheric string ensemble. The composition is in the key of D minor with a 4/4 time signature and a slow Adagio tempo. The 8-measure progression Dm - Bb - Gm - A7, followed by Dm - Bb - Gm - A7 - Dm, creates a structured emotional arc that moves from a sense of unresolved longing toward a poignant and quiet finality.
- Tamaño del MIDI generado: 523 bytes
💾 Guardado en: outputs/evaluacion-notebook/text2midi_resultado.mid


## 6. Prueba 2: Flujo MidiLLM (Batch / Best-of-N Search)

Utilizaremos el perfil `midillm-fast`, que genera múltiples secuencias candidatas en bloque (batch) y luego usa el evaluador para elegir la mejor (Best-of-N).

In [5]:
print("\n" + "=" * 50)
print("🎹 INICIANDO PRUEBA 2: MidiLLM (Best-of-N)")
print("=" * 50)

# Cargar el perfil midillm
profile_mllm = get_profile("midillm-fast")

# Generar MIDI
result_mllm = pipeline.generate(text=user_prompt, profile=profile_mllm)

print("\n✨ Resultados MidiLLM:")
print(
    f"- Traducción (Technical Prompt) usada por el generador:\n{result_mllm.technical_prompt}"
)
print(f"- Tamaño del MIDI generado: {len(result_mllm.midi_bytes)} bytes")

# Guardar
out_path_mllm = output_dir / "midillm_resultado.mid"
with open(out_path_mllm, "wb") as f:
    f.write(result_mllm.midi_bytes)
print(f"💾 Guardado en: {out_path_mllm}")

2026-05-25 16:47:38,766 - pipeline - INFO - Generating MIDI for: 'Una melodía de piano melancólica y lenta, con un a...' with profile GenerationProfile(token_batch_size=500, num_beams=1, top_k=1, max_tokens=2046, random_seed=None, clap_weight=0.4, key_weight=0.3, note_weight=0.3, strict_instruments=False, generator_type='midillm', num_outputs=4)



🎹 INICIANDO PRUEBA 2: MidiLLM (Best-of-N)


2026-05-25 16:47:38,869 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/slseanwu/MIDI-LLM_Llama-3.2-1B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 16:47:38,882 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/slseanwu/MIDI-LLM_Llama-3.2-1B/6eeae5e9fd3959a61b8b6690787bff38d1ec3664/config.json "HTTP/1.1 200 OK"
2026-05-25 16:47:38,985 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/slseanwu/MIDI-LLM_Llama-3.2-1B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 16:47:38,999 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/slseanwu/MIDI-LLM_Llama-3.2-1B/6eeae5e9fd3959a61b8b6690787bff38d1ec3664/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-25 16:47:39,216 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/slseanwu/MIDI-LLM_Llama-3.2-1B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


Loading weights:   0%|          | 0/147 [00:00<?, ?it/s]

2026-05-25 16:47:46,167 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/slseanwu/MIDI-LLM_Llama-3.2-1B/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 16:47:46,180 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/slseanwu/MIDI-LLM_Llama-3.2-1B/6eeae5e9fd3959a61b8b6690787bff38d1ec3664/generation_config.json "HTTP/1.1 200 OK"
2026-05-25 16:47:47,323 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-05-25 16:48:16,015 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-26b-a4b-it:generateContent "HTTP/1.1 200 OK"
2026-05-25 16:48:16,017 - use_cases.best_of_n_search - INFO - Starting Best-of-N search with num_outputs=4, prompt='A cinematic classical track with a melancholic and...'
2026-05-25 16:48:40,839 - use_cases.best_of_n_search - INFO - Generated 4 sequences
2026-05-25 16:48:41,716 - use_cases.best_of_n_search - INFO - Best-of-N se


✨ Resultados MidiLLM:
- Traducción (Technical Prompt) usada por el generador:
A cinematic classical track with a melancholic and somber vibe, featuring a delicate felt piano melody accompanied by a soft, legato string ensemble. The song is in the key of A minor with a 4/4 time signature and a slow Adagio tempo. The 8-measure chord progression Am - Dm - G - E, followed by Am - Dm - F - E7 - Am, guides the listener through a journey of quiet sadness, moving from a state of unresolved longing toward a gentle, final resolution.
- Tamaño del MIDI generado: 2105 bytes
💾 Guardado en: outputs/evaluacion-notebook/midillm_resultado.mid


## 7. Conclusiones y Revisión

* Revisa las traducciones (technical_prompts) para asegurar que el traductor interpretó correctamente la intención.
* Escucha los archivos MIDI generados (`outputs/evaluacion-notebook/*.mid`) y compáralos según los criterios de la investigación.